<a href="https://colab.research.google.com/github/dennzii/Low-Memory-Difusion-Based-Virtual-Try-On-via-Quantized-Stable-Difusion-Backbones/blob/main/concat_selfattn_sd15_last_LPIPS_0.0947_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bitsandbytes

In [ ]:
!pip install lpips

In [ ]:
!pip install xformers

In [ ]:
import kagglehub
!pip install kagglehub

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch
from diffusers import DiffusionPipeline
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#for faster mat mul operations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub
import os

# Download latest version of the dataset
path = kagglehub.dataset_download("tinkukalluri/zalando-hd-resized")

# The dataset_base_path will be the path returned by kagglehub
dataset_base_path = path

print(f"Dataset indirildi ve extract edildi: {dataset_base_path}")

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
import numpy as np
import cv2


class VITONDataset(Dataset):

    def __init__(self, dataset_path, image_size=(512, 384)):
        self.dataset_path = dataset_path
        self.image_size = image_size

        self.file_names = sorted(os.listdir(os.path.join(dataset_path, "image")))

        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.file_names)

    def load_rgb_tensor(self, path):
        # OpenCV ile oku ve BGR'dan RGB'ye çevir
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # cv2.resize hedef boyutu (W, H) olarak bekler
        img_resized = cv2.resize(img, (self.image_size[1], self.image_size[0]), interpolation=cv2.INTER_AREA)
        return self.to_tensor(img_resized)  # (3,H,W)

    def concat_images_tensor(self, cloth, image):
        # cloth + image → width concat
        return torch.cat([cloth, image], dim=2)  # (3, H, 2W)

    def zero_pad_left_tensor(self, mask):
        # mask: (1,H,W) → (1,H,2W)
        B, H, W = mask.shape
        pad = torch.zeros((1, H, W), device=mask.device)
        return torch.cat([pad, mask], dim=2)

    def load_agnostic_mask(self, mask_path):
        # Maskeyi Grayscale olarak oku
        img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        # INTER_NEAREST ile yeniden boyutlandır
        img_resized = cv2.resize(img, (self.image_size[1], self.image_size[0]), interpolation=cv2.INTER_NEAREST)

        # Değeri 0-1 aralığına çek ve threshold uygula
        mask_array = np.array(img_resized, dtype=np.float32) / 255.0
        mask_array[mask_array < 0.5] = 0.0
        mask_array[mask_array >= 0.5] = 1.0

        return torch.from_numpy(mask_array).unsqueeze(0)  # (1, H, W)

    def __getitem__(self, idx):

        file_name = self.file_names[idx]

        cloth_path = os.path.join(self.dataset_path, "cloth", file_name)
        image_path = os.path.join(self.dataset_path, "image", file_name)

        # Mask dosya ismini bulma (genelde .png veya _mask.png olabilir)
        mask_name = file_name.replace('.jpg', '.png')
        mask_path = os.path.join(self.dataset_path, "agnostic-mask", mask_name)
        if not os.path.exists(mask_path):
            mask_path = os.path.join(self.dataset_path, "agnostic-mask", file_name)
        if not os.path.exists(mask_path):
            mask_path = os.path.join(self.dataset_path, "agnostic-mask", file_name.replace('.jpg', '_mask.png'))

        # load tensors
        cloth = self.load_rgb_tensor(cloth_path)
        image = self.load_rgb_tensor(image_path)

        # concat
        concat_tensor = self.concat_images_tensor(cloth, image)

        # Hazır agnostic-mask dizininden maskeyi yükle
        mask_512 = self.load_agnostic_mask(mask_path)

        mask_pad = self.zero_pad_left_tensor(mask_512)

        return concat_tensor, mask_pad


In [ ]:
from google.colab import drive
import os

save_dir = "/content/drive/MyDrive/VTON_Project/checkpoints"
os.makedirs(save_dir, exist_ok=True)

In [ ]:


# dataset_base_path is now set by the kagglehub download in the previous cell

train_ds = VITONDataset(dataset_path=os.path.join(dataset_base_path,"train"))
test_ds = VITONDataset(dataset_path=os.path.join(dataset_base_path,"test"))

train_loader = DataLoader(train_ds,batch_size=128,shuffle=True,num_workers=8,
    pin_memory=True,
    persistent_workers=True)

test_loader = DataLoader(test_ds,batch_size=1,shuffle=False,num_workers=2)

In [ ]:
from diffusers.models.autoencoders import AutoencoderKL

vae = AutoencoderKL.from_pretrained("stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="vae").to(device)
vae.eval()

In [ ]:
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
noise_scheduler = DDPMScheduler.from_pretrained("stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="scheduler")

In [ ]:
import torch
import torch.nn as nn
import gc
from diffusers import UNet2DConditionModel
from diffusers.models.attention import BasicTransformerBlock

# Dummy (Boş) Cross-Attention Sınıfı
class DummyCrossAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # Hiçbir parametre içermez, VRAM harcamaz.

    def forward(self, hidden_states, encoder_hidden_states=None, attention_mask=None, **kwargs):
        # İşlem yapmadan gelen veriyi doğrudan geri döndür
        return hidden_states

# Kalıcı Silme Fonksiyonu
def strip_cross_attention_permanently(unet_model):
    removed_params = 0
    for name, module in unet_model.named_modules():
        if isinstance(module, BasicTransformerBlock):
            # attn2'yi (Cross-Attention) tamamen Dummy katmanla değiştir
            if hasattr(module, 'attn2') and module.attn2 is not None:
                removed_params += sum(p.numel() for p in module.attn2.parameters())
                module.attn2 = DummyCrossAttention()

            # norm2'yi (Cross-Attention öncesi normalizasyon) Identity ile değiştir
            if hasattr(module, 'norm2') and module.norm2 is not None:
                removed_params += sum(p.numel() for p in module.norm2.parameters())
                module.norm2 = nn.Identity()

    # garbage collector çalıştır ve VRAM'i temizle
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Başarıyla {removed_params / 1e6:.2f}M parametre silindi ve VRAM'den temizlendi!")
    return unet_model

In [ ]:
import torch.nn as nn
from diffusers import UNet2DConditionModel

unet = UNet2DConditionModel.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="unet"
)

unet = strip_cross_attention_permanently(unet)

unet.to(device)

# önce freeze et.
for param in unet.parameters():
    param.requires_grad = False

# sadece attn1 layerlarını aç
for name, module in unet.named_modules():
    if "attn1" in name:
        for param in module.parameters():
            param.requires_grad = True

# kontrol
trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print("Trainable params:", trainable)

In [ ]:
import torch

unet.enable_gradient_checkpointing()
#unet.enable_xformers_memory_efficient_attention()
# PyTorch 2.0 kernel fusion ile modelin derlenip hızlandırılması

!PYTORCH_ALLOC_CONF=expandable_segments:True

### 1. Latent Caching: Drive'dan Yükleme (Alternatif)
Bu adımı, eğer önceden oluşturulmuş latent dosyalarınızı Drive'dan yüklemek istiyorsanız kullanın. Aksi takdirde, bir sonraki "Latent Caching: VAE'den Geçirip Diske Kaydetme" adımını çalıştırabilirsiniz.

### 1. Latent Caching: VAE'den Geçirip Diske Kaydetme
Bu işlem veri setinizin büyüklüğüne göre biraz zaman alabilir, ancak sadece **bir kere** yapılacaktır.

In [ ]:
import os
import torch
import torch.nn.functional as F
from tqdm import tqdm
import shutil

# Latent tensörlerin kaydedileceği ana dizin
latent_save_dir = "/content/latents"
os.makedirs(os.path.join(latent_save_dir, "train"), exist_ok=True)
os.makedirs(os.path.join(latent_save_dir, "test"), exist_ok=True)

@torch.no_grad()
def cache_latents(dataloader, split_name):
    vae.eval()
    for i, batch in enumerate(tqdm(dataloader, desc=f"{split_name} Caching")):
        target_images = batch[0].to(device)
        masks = batch[1].to(device)

        # Normalizasyon [-1, 1]
        target_images = target_images * 2.0 - 1.0
        masked_images = target_images * (1 - masks)

        # Unconditional Masked Images
        uncond_masked_images = masked_images.clone()
        uncond_masked_images[:, :, :, :384] = -1.0

        with torch.amp.autocast("cuda"):
            latents = vae.encode(target_images).latent_dist.sample() * vae.config.scaling_factor
            masked_latents = vae.encode(masked_images).latent_dist.sample() * vae.config.scaling_factor
            uncond_masked_latents = vae.encode(uncond_masked_images).latent_dist.sample() * vae.config.scaling_factor

        latent_masks = F.interpolate(masks, size=latents.shape[-2:], mode="nearest-exact")

        for b_idx in range(latents.size(0)):
            file_idx = i * dataloader.batch_size + b_idx
            file_path = os.path.join(latent_save_dir, split_name, f"{file_idx}.pt")

            torch.save({
                "latent": latents[b_idx].cpu(),
                "masked_latent": masked_latents[b_idx].cpu(),
                "uncond_masked_latent": uncond_masked_latents[b_idx].cpu(),
                "mask": latent_masks[b_idx].cpu()
            }, file_path)

print("Yerel diske Latent Caching başlatılıyor...")
cache_latents(train_loader, "train")
cache_latents(test_loader, "test")
print(f" Caching tamamlandı! Dosyalar burada: {latent_save_dir}")

# Zipleyip Drive'a yedekle (yeni isimle)
drive_backup_path = "/content/drive/MyDrive/VTON_Project/latents_inter_area.zip"
print("\nLatents klasörü zipe dönüştürülüyor...")
shutil.make_archive("/content/latents_inter_area", 'zip', latent_save_dir)

print(f"Zip dosyası Drive'a kopyalanıyor: {drive_backup_path}")
shutil.copy("/content/latents_inter_area.zip", drive_backup_path)
print(" Yedekleme tamamlandı!")


In [ ]:
import os

# Drive'daki latent zip dosyasının yolu
drive_zip_path = "/content/drive/MyDrive/VTON_Project/latents_inter_area.zip" # Burayı kendi zip dosyanızın yoluna göre düzenleyin!

# Latent dosyalarının extract edileceği yerel dizin
local_latents_dir = "/content/latents"

print(f"Latent dosyaları Drive'dan kopyalanıp extract ediliyor: {drive_zip_path} -> {local_latents_dir}")

# Hedef dizini oluştur (zaten varsa sorun değil)
!mkdir -p {local_latents_dir}/train
!mkdir -p {local_latents_dir}/test

# Zip dosyasını geçici olarak Colab ortamına kopyala
!cp "{drive_zip_path}" "/content/temp_latents.zip"

# Zip dosyasını hedef dizine aç
# `unzip -q` sessiz modda çalışır, `-d` belirtilen dizine açar
!unzip -q "/content/temp_latents.zip" -d "{local_latents_dir}"

# Kopyalanan zip dosyasını temizle
!rm "/content/temp_latents.zip"

print(f" Latent dosyaları başarıyla Drive'dan yüklendi ve '{local_latents_dir}' dizinine açıldı.")
print(" Eğer bu adımı kullandıysanız, lütfen bir sonraki 'Latent Caching: VAE'den Geçirip Diske Kaydetme' başlığındaki hücreyi atlayın.")

### 2. Cached Dataset ve DataLoader
Artık resim okumak yerine bu kaydettiğimiz latent dosyaları okuyacağız.

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader

# Artık Drive'dan kopyalamaya gerek yok, doğrudan /content/latents kullanıyoruz
local_latents_dir = "/content/latents"

class CachedVITONDataset(Dataset):
    def __init__(self, latent_dir):
        self.latent_dir = latent_dir
        self.files = sorted(
            [f for f in os.listdir(latent_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('.')[0])
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.latent_dir, self.files[idx])
        data = torch.load(path, map_location="cpu", weights_only=True)
        return data["latent"], data["masked_latent"], data["uncond_masked_latent"], data["mask"]

cached_train_ds = CachedVITONDataset(os.path.join(local_latents_dir, "train"))
cached_train_loader = DataLoader(
    cached_train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    drop_last=False
)

print(f" Önbelleğe alınmış yerel eğitim verisi sayısı: {len(cached_train_ds)}")

### 3. VAE'siz Yeni Eğitim Döngüsü
Bu döngüde VAE encode işlemleri olmadığı için epoch süreleri inanılmaz derecede kısalacaktır.

In [ ]:
import os
import torch

# Lütfen buraya Drive'ınızda bulunan en son checkpoint dosyasının tam adını yazın
checkpoint_filename = "dream_cached_interarea_epoch_58_0.025408.pt"
checkpoint_path = os.path.join(save_dir, checkpoint_filename)

load_checkpoint = True # Devam etmek için True yaptık

start_epoch = 0
if load_checkpoint and os.path.exists(checkpoint_path):
    print(f"Checkpoint yükleniyor: {checkpoint_path}")

    # Modeli geçici olarak CPU'ya alıp yüklemek VRAM dalgalanmalarını önleyebilir
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

    # UNet ağırlıklarını yükle (cross_attention layerları silindiği için strict=False önemli)
    unet.load_state_dict(checkpoint["unet"], strict=False)
    print(" UNet ağırlıkları başarıyla yüklendi.")

    # Optimizer durumunu atlıyoruz çünkü bir sonraki hücrede baştan tanımlanacak
    print("ℹ Optimizer durumu atlandı (bir sonraki hücrede yeni optimizer oluşturulacak).")

    loaded_epoch = checkpoint.get("epoch", 58)
    loaded_loss = checkpoint.get("loss", "Bilinmiyor")
    start_epoch = loaded_epoch + 1
    print(f" Yüklenen Model -> Epoch: {loaded_epoch}, Loss: {loaded_loss}")
    print(f"Eğitim {start_epoch}. epoch'tan başlayacak.")

else:
    print("ℹ Checkpoint yüklenmeyecek. Model sıfırdan eğitilecek.")

# Modeli cihaza (GPU) geri gönder
unet.to(device)
print("Model eğitime hazır!")

In [ ]:
import math # For math.ceil

# Sabit öğrenme oranı
learning_rate = 1e-5

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=learning_rate
)

# Toplam epoch sayısı
epochs = 250

num_batches_per_epoch = math.ceil(len(cached_train_ds) / cached_train_loader.batch_size)

print(f"Optimizer sabit {learning_rate} learning rate ile ayarlandı.")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from tqdm import tqdm
import torch
import torch.nn.functional as F
import shutil
import os

scaler = torch.amp.GradScaler("cuda")
unet.train()

best_loss = float("inf")
# start_epoch bir önceki hücrede belirleniyor

# Toplam adım limitini belirliyoruz (epochs = 250 olarak tanımlı)
decay_steps_limit = num_batches_per_epoch * epochs
past_steps = start_epoch * num_batches_per_epoch

global_batch_step = past_steps

for epoch_num_for_logging in range(start_epoch, epochs):
    epoch_loss = 0
    pbar = tqdm(cached_train_loader, desc=f"Epoch {epoch_num_for_logging}/{epochs - 1}")

    # Dataloader'dan gelen 3. elemanı (eski uncond_latents) '_' ile yoksayıyoruz
    for latents, masked_latents, _, masks in pbar:

        # Cihaza gönder
        latents = latents.to(device)
        masked_latents = masked_latents.to(device)
        masks = masks.to(device)

        # --- CFG DÜZELTMESİ BAŞLANGICI ---
        # Koşulsuz (uncond) latent'i siyah resim kodlamak yerine doğrudan latent uzayında sıfırla
        uncond_masked_latents = masked_latents.clone()
        uncond_masked_latents[:, :, :, :48] = 0.0
        # --- CFG DÜZELTMESİ BİTİŞİ ---

        # CFG Dropout: %10 ihtimalle condition'ı unconditioned (kıyafet silinmiş) latent'e çevir
        drop_mask = torch.rand(latents.shape[0], device=device) < 0.1

        final_masked_latents = torch.where(
            drop_mask.view(-1, 1, 1, 1),
            uncond_masked_latents,
            masked_latents
        )

        # Noise ve Timesteps ayarları
        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],),
            device=device
        ).long()

        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # --- DREAM: first pass (teacher) ---
        with torch.no_grad():
            with torch.amp.autocast("cuda"):
                initial_input = torch.cat([noisy_latents, masks, final_masked_latents], dim=1)
                initial_noise_pred = unet(initial_input, timesteps, encoder_hidden_states=None).sample

        alpha_bar_t = noise_scheduler.alphas_cumprod[timesteps].view(-1, 1, 1, 1)
        lambda_t  = (1.0 - alpha_bar_t).sqrt() ** 10

        delta_eps = noise - initial_noise_pred
        noisy_latents_dream = noisy_latents + lambda_t * delta_eps

        # --- DREAM: second pass (student) ---
        model_input = torch.cat([noisy_latents_dream, masks, final_masked_latents], dim=1)

        with torch.amp.autocast("cuda"):
            noise_pred = unet(model_input, timesteps, encoder_hidden_states=None).sample

            # DREAM Hedefi
            target = noise + lambda_t * delta_eps

            loss = F.mse_loss(noise_pred, target)

        # Backprop
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        global_batch_step += 1

        epoch_loss += loss.item()
        pbar.set_postfix({"loss": loss.item(), "lr": optimizer.param_groups[0]["lr"]})

    avg_loss = epoch_loss / len(cached_train_loader)
    print(f"\nEpoch {epoch_num_for_logging} Ortalama Loss: {avg_loss:.6f}, Current LR: {optimizer.param_groups[0]['lr']:.7f}")

    file_pth = f"dream_cached_interarea_epoch_{epoch_num_for_logging}_{avg_loss:.6f}.pt"
    temp_save_path = os.path.join("/content/", file_pth)

    torch.save({
        "epoch": epoch_num_for_logging,
        "unet": unet.state_dict(),
        "optimizer": optimizer.state_dict(),
        "loss": avg_loss,
    }, temp_save_path)

    shutil.copy(temp_save_path, os.path.join(save_dir, file_pth))
    os.remove(temp_save_path)

In [ ]:
from diffusers import DDIMScheduler
import torch.nn.functional as F
from tqdm import tqdm
import torch

# Orijinal makaleye uygun olarak çıkarım için DDIM scheduler kullanıyoruz
inference_scheduler = DDIMScheduler.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="scheduler"
)

@torch.inference_mode()
def run_inference(batch, num_steps=50, cfg=2.5, verbose=True):

    concat_tensor, mask_tensor = batch
    concat_tensor = concat_tensor.to(device)
    mask_tensor = mask_tensor.to(device)

    # Artık direkt batch boyutunu (B, C, H, W) koruyoruz
    image_batch = concat_tensor.clone()
    mask_batch = mask_tensor.clone()

    # normalize (-1,1) vae böyle bir girdi alıyor.
    image_batch = image_batch * 2.0 - 1.0

    # her ihtimale karşı mask'ı binary hale getirelim
    mask_batch = (mask_batch > 0.5).float()

    # cond masked
    masked_image = image_batch * (1 - mask_batch)

    # ---------------------------------
    # --- CFG DÜZELTMESİ BAŞLANGICI ---
    # Piksel uzayında -1.0 yapma işlemini iptal ettik.

    # VAE
    image_latents = vae.encode(image_batch).latent_dist.sample() * vae.config.scaling_factor
    masked_latents = vae.encode(masked_image).latent_dist.sample() * vae.config.scaling_factor

    # Koşulsuz latent'i tıpkı eğitimdeki gibi doğrudan latent uzayında sıfırlıyoruz.
    uncond_masked_latents = masked_latents.clone()
    uncond_masked_latents[:, :, :, :48] = 0.0
    # --- CFG DÜZELTMESİ BİTİŞİ ---
    # ---------------------------------

    mask_latent = F.interpolate(mask_batch, size=masked_latents.shape[-2:], mode="nearest")

    # diffusion
    inference_scheduler.set_timesteps(num_steps, device=device)
    timesteps = inference_scheduler.timesteps

    # Inference'ta inpainting için saf gürültü yerine masked_latents üzerine gürültü ekliyoruz.
    # scheduler'ın ilk adımı (en yüksek timestep) için add_noise kullanılır.
    noise = torch.randn_like(masked_latents)
    latents = inference_scheduler.add_noise(masked_latents, noise, timesteps[:1])

    # denoise
    # leave=False parametresi ise bar aktif olsa bile tamamlanınca ekrandan silinmesini sağlar.
    for t in tqdm(timesteps, disable=not verbose, leave=False):

        # Scheduler'ın beklentisine göre latent'leri scale ediyoruz
        latent_model_input = inference_scheduler.scale_model_input(latents, t)

        # scale edilmiş latent_model_input'u kullanıyoruz
        cond_input = torch.cat([latent_model_input, mask_latent, masked_latents], dim=1)
        uncond_input = torch.cat([latent_model_input, mask_latent, uncond_masked_latents], dim=1)

        # FP32 inference için autocast kaldırıldı
        cond_pred = unet(cond_input, t, encoder_hidden_states=None).sample
        uncond_pred = unet(uncond_input, t, encoder_hidden_states=None).sample

        # CFG 2.5-3.5 arası en iyi kalite veriyor. catvton makalesine göre
        noise_pred = uncond_pred + cfg * (cond_pred - uncond_pred)

        latents = inference_scheduler.step(noise_pred, t, latents).prev_sample

    # DECODE İŞLEMİNİ TÜM RESİM İÇİN YAP (Renk/Parlaklık kaymasını önlemek için)
    latents = latents / vae.config.scaling_factor
    output_full = vae.decode(latents).sample
    output_full = (output_full / 2 + 0.5).clamp(0, 1)

    # SADECE SAĞ TARAFI KES (Pixel Space)
    _, _, H_img, W_img = image_batch.shape
    right_image = image_batch[:, :, :, W_img // 2:]
    right_mask = mask_batch[:, :, :, W_img // 2:]
    output_right = output_full[:, :, :, W_img // 2:]

    original_right_01 = (right_image / 2 + 0.5).clamp(0, 1)
    final_output = original_right_01 * (1 - right_mask) + output_right * right_mask

    # Artık sadece sağ taraf (üretilen kişi) döndürülüyor
    return final_output.float()

In [ ]:
import torch
import matplotlib.pyplot as plt
import os

# Modeli yükle
ckpt_path = os.path.join(save_dir+"/last_ckpts", "dream_cached_interarea_epoch_110_0.025027.pt")
print(f"Model yükleniyor: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location="cpu")
unet.load_state_dict(ckpt["unet"], strict=False)
unet.to(device)
print("Model yüklendi!")

# Tek bir örnek seçelim (Örnek olarak indeks 10 ve 20)
person_idx = 55
cloth_idx = 490

person_concat, person_mask = test_ds[person_idx]
cloth_concat, _ = test_ds[cloth_idx]

mixed_concat = person_concat.clone()
mixed_concat[:, :, :384] = cloth_concat[:, :, :384] # Kıyafeti değiştir

mixed_item = (mixed_concat.unsqueeze(0), person_mask.unsqueeze(0))

# Test edilecek CFG değerleri
cfg_values = [2.5, 3.0, 3.5, 4.0, 5.0, 6.0]

# Figür oluştur (Kişi, Kıyafet + CFG çıktıları)
fig, axes = plt.subplots(1, len(cfg_values) + 2, figsize=(3.5 * (len(cfg_values) + 2), 6))

person_img = person_concat[:, :, 384:].permute(1, 2, 0).cpu().numpy()
cloth_img = cloth_concat[:, :, :384].permute(1, 2, 0).cpu().numpy()

axes[0].imshow(person_img)
axes[0].set_title("Original Person")
axes[0].axis("off")

axes[1].imshow(cloth_img)
axes[1].set_title("Target Cloth")
axes[1].axis("off")

for i, cfg in enumerate(cfg_values):
    print(f"CFG {cfg} için çıkarım yapılıyor...")
    output = run_inference(mixed_item, num_steps=50, cfg=cfg, verbose=False)
    out_img = output.squeeze(0).permute(1, 2, 0).cpu().numpy()

    ax = axes[i + 2]
    ax.imshow(out_img)
    ax.set_title(f"CFG: {cfg}")
    ax.axis("off")

plt.tight_layout()
plt.savefig("cfg_comparison.png", dpi=600)
plt.show()

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from diffusers import UNet2DConditionModel

unet = UNet2DConditionModel.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="unet"
)

unet = strip_cross_attention_permanently(unet)

# strict=False
ckpt_path = os.path.join(save_dir+"/last_ckpts", "dream_cached_interarea_epoch_110_0.025027.pt")
ckpt = torch.load(ckpt_path, map_location="cpu")
unet.load_state_dict(ckpt["unet"], strict=False)

# Cihaza gönder
unet.to("cuda")
print("UNet kullanıma hazır!")

# Inference Kısmı

In [ ]:
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

In [ ]:
from torchvision.utils import save_image
from tqdm import tqdm

# 3 farklı kişi ve 3 farklı kıyafet indeksi
person_indices = [7]
cloth_indices = [67]

for idx_num, (p_idx, c_idx) in enumerate(zip(person_indices, cloth_indices), start=1):
    print(f"{idx_num}. kombinasyon hazırlanıyor (Kişi İndeksi: {p_idx}, Kıyafet İndeksi: {c_idx})...")

    # Dataset'ten verileri doğrudan çek (DataLoader'da dönmekten çok daha hızlı)
    person_concat, person_mask = test_ds[p_idx]
    cloth_concat, _ = test_ds[c_idx]

    # Kıyafeti (sol taraf) kişinin verisine ekle
    mixed_concat = person_concat.clone()
    mixed_concat[:, :, :384] = cloth_concat[:, :, :384]

    # Model batched veri beklediği için unsqueeze(0) ile batch boyutu ekliyoruz
    mixed_item = (mixed_concat.unsqueeze(0), person_mask.unsqueeze(0))

    # Inference (Sonuç sadece sağ tarafı, yani üretilen kişiyi döndürür)
    output = run_inference(mixed_item, num_steps=50, cfg=4.0, verbose=False)

    # Orijinal kıyafet ve kişi resimlerini ayır
    cloth_img = cloth_concat[:, :, :384]
    person_img = person_concat[:, :, 384:]

    # Resimleri kaydet
    save_image(cloth_img, fp=f"/content/cloth_{idx_num}.png")
    save_image(person_img, fp=f"/content/person_{idx_num}.png")
    save_image(output, fp=f"/content/result_{idx_num}.png")

    print(f" Kaydedildi: cloth_{idx_num}.png, person_{idx_num}.png, result_{idx_num}.png\n")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Load the generated image
output_image_path = "/content/output.png"
output_image = Image.open(output_image_path)

# Display the image
plt.figure(figsize=(7, 7))
plt.imshow(output_image)
plt.axis('off')
plt.title('Generated Output Image')
plt.show()

In [ ]:
import os
from torchvision.utils import save_image
from diffusers import DDIMScheduler, DPMSolverMultistepScheduler
import torch

# Grid search klasörü
grid_save_folder = "/content/grid_search_outputs"
os.makedirs(grid_save_folder, exist_ok=True)

# Hedef kişi ve kıyafet
person_idx = 43
cloth_idx = 1325

# Veriyi hazırla
person_item = None
cloth_item = None

for i, batch in enumerate(test_loader):
    if i == person_idx:
        person_item = batch
    if i == cloth_idx:
        cloth_item = batch
    if person_item is not None and cloth_item is not None:
        break

mixed_concat = person_item[0].clone()
mixed_concat[:, :, :, :384] = cloth_item[0][:, :, :, :384]
mixed_item = (mixed_concat, person_item[1])

# Test edilecek parametreler
cfg_scales = [2.5, 3.5, 4.5, 5.5]
steps_list = [20, 25, 30]
scheduler_names = [ "DPM++2M"]

# Orijinal scheduler'ı yedekle
original_scheduler = inference_scheduler

print(f"Grid search başlıyor... Toplam kombinasyon: {len(cfg_scales) * len(steps_list) * len(scheduler_names)}")

for sched_name in scheduler_names:
    # İlgili scheduler'ı yükle
    if sched_name == "DDIM":
        inference_scheduler = DDIMScheduler.from_pretrained(
            "stable-diffusion-v1-5/stable-diffusion-inpainting",
            subfolder="scheduler"
        )
    elif sched_name == "DPM++2M":
        inference_scheduler = DPMSolverMultistepScheduler.from_pretrained(
            "stable-diffusion-v1-5/stable-diffusion-inpainting",
            subfolder="scheduler"
        )

    for steps in steps_list:
        for cfg in cfg_scales:
            print(f"Test ediliyor -> Scheduler: {sched_name}, Steps: {steps}, CFG: {cfg}")

            # Inference
            output = run_inference(mixed_item, num_steps=steps, cfg=cfg, verbose=False)

            # Kaydet
            filename = f"{sched_name}_steps{steps}_cfg{cfg}.png"
            filepath = os.path.join(grid_save_folder, filename)
            save_image(output, fp=filepath)

# Orijinal scheduler'ı geri yükle
inference_scheduler = original_scheduler
print(f"\n Tüm testler bitti! Sonuçlar '{grid_save_folder}' klasörüne kaydedildi.")

In [ ]:
import os
from torchvision.utils import save_image
from diffusers import DPMSolverMultistepScheduler
import torch

# Düşük stepler için yeni klasör
grid_save_folder_low_steps = "/content/grid_search_low_steps"
os.makedirs(grid_save_folder_low_steps, exist_ok=True)

# Daha düşük step ve benzer CFG değerleri
cfg_scales = [2.5, 3.5, 4.5]
steps_list = [15, 20, 25, 30]

# Orijinal scheduler'ı yedekle
original_scheduler = inference_scheduler

# DPM++ 2M Karras'ı yükle
inference_scheduler = DPMSolverMultistepScheduler.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="scheduler",
    use_karras_sigmas=True # Karras metodunu aktif ediyoruz
)

print(f"Düşük Step Grid Search (DPM++ 2M Karras) başlıyor... Toplam kombinasyon: {len(cfg_scales) * len(steps_list)}")

for steps in steps_list:
    for cfg in cfg_scales:
        print(f"Test ediliyor -> Steps: {steps}, CFG: {cfg}")

        # Inference (Daha önceki mixed_item kullanılıyor)
        output = run_inference(mixed_item, num_steps=steps, cfg=cfg, verbose=False)

        # Kaydet
        filename = f"DPM2M_Karras_steps{steps}_cfg{cfg}.png"
        filepath = os.path.join(grid_save_folder_low_steps, filename)
        save_image(output, fp=filepath)

# Orijinal scheduler'ı geri yükle
inference_scheduler = original_scheduler
print(f"\nTüm testler bitti! Sonuçlar '{grid_save_folder_low_steps}' klasörüne kaydedildi.")

In [ ]:
import pandas as pd
import torch
import lpips
from tqdm import tqdm
from diffusers import DDIMScheduler, DPMSolverMultistepScheduler
import os
from google.colab import files
from torch.utils.data import DataLoader

# Initialize Schedulers
ddim_scheduler = DDIMScheduler.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="scheduler"
)
karras_scheduler = DPMSolverMultistepScheduler.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="scheduler", use_karras_sigmas=True
)

configs = [
    {"name": "DDIM_50_CFG_2.5", "scheduler": ddim_scheduler, "steps": 50, "cfg": 2.5},
    {"name": "DDIM_50_CFG_3.0", "scheduler": ddim_scheduler, "steps": 50, "cfg": 3.0},
    {"name": "DDIM_50_CFG_3.5", "scheduler": ddim_scheduler, "steps": 50, "cfg": 3.5},
     {"name": "DDIM_50_CFG_3.5", "scheduler": ddim_scheduler, "steps": 50, "cfg": 4.0},
    {"name": "Karras_25_CFG_2.5", "scheduler": karras_scheduler, "steps": 25, "cfg": 2.5},
    {"name": "Karras_25_CFG_3.0", "scheduler": karras_scheduler, "steps": 25, "cfg": 3.0},
    {"name": "Karras_25_CFG_3.5", "scheduler": karras_scheduler, "steps": 25, "cfg": 3.5},
     {"name": "Karras_25_CFG_3.5", "scheduler": karras_scheduler, "steps": 25, "cfg": 4.0},
    {"name": "Karras_30_CFG_2.5", "scheduler": karras_scheduler, "steps": 30, "cfg": 2.5},
    {"name": "Karras_30_CFG_3.0", "scheduler": karras_scheduler, "steps": 30, "cfg": 3.0},
    {"name": "Karras_30_CFG_3.5", "scheduler": karras_scheduler, "steps": 30, "cfg": 3.5},
    {"name": "Karras_30_CFG_3.5", "scheduler": karras_scheduler, "steps": 30, "cfg": 4.0}
]

# Load LPIPS model (AlexNet is faster for evaluation)
if 'loss_fn_alex' not in globals():
    loss_fn_alex = lpips.LPIPS(net='alex').to(device)
    loss_fn_alex.eval()

eval_batch_size = 4
eval_loader = DataLoader(test_ds, batch_size=eval_batch_size, shuffle=False, num_workers=4)
num_samples = 64

all_scores = {cfg["name"]: [] for cfg in configs}

global inference_scheduler
original_scheduler = inference_scheduler

for config in configs:
    inference_scheduler = config["scheduler"]
    print(f"Testing {config['name']}...")

    current_scores = []

    with torch.no_grad():
        for i, batch in enumerate(tqdm(eval_loader, desc=config["name"])):
            if i * eval_batch_size >= num_samples:
                break

            concat_tensor = batch[0].to(device)
            W2 = concat_tensor.shape[3]
            W = W2 // 2
            gt_image_01 = concat_tensor[:, :, :, W:]

            # Inference
            output_01 = run_inference(batch, num_steps=config["steps"], cfg=config["cfg"], verbose=False)

            gt_lpips = gt_image_01 * 2.0 - 1.0
            output_lpips = output_01 * 2.0 - 1.0

            # Calculate LPIPS for the batch without reduction
            lpips_vals = loss_fn_alex(output_lpips, gt_lpips).squeeze().tolist()

            if isinstance(lpips_vals, float):
                lpips_vals = [lpips_vals]

            current_scores.extend(lpips_vals)

    all_scores[config["name"]] = current_scores

# Restore original scheduler
inference_scheduler = original_scheduler

# Calculate averages
avg_data = {"Config": [], "Average_LPIPS": []}
for config in configs:
    avg_val = sum(all_scores[config["name"]]) / len(all_scores[config["name"]])
    avg_data["Config"].append(config["name"])
    avg_data["Average_LPIPS"].append(avg_val)

# Create and display DataFrame for averages
df_avg = pd.DataFrame(avg_data)
display(df_avg)

# Save and download
csv_filename = "lpips_avg_comparison.csv"
df_avg.to_csv(csv_filename, index=False)
print(f"Saved average results to {csv_filename}")
files.download(csv_filename)

In [ ]:
# @title
import os
from torchvision.utils import save_image

def try_on_custom_cloth(person_idx, cloth_path, num_steps=50, cfg=3.0):
    # 1. Kişinin verilerini test setinden al (maske ve kişi resmi lazım)
    person_concat, person_mask = test_ds[person_idx]

    # 2. Kendi kıyafet resmimizi yükleyip tensor'a çevirelim
    # (Dataset'teki hazır fonksiyon otomatik olarak 512x384 boyutuna getirir ve normalize eder)
    cloth_tensor = test_ds.load_rgb_tensor(cloth_path)


    mixed_concat = person_concat.clone()


    mixed_concat[:, :, :384] = cloth_tensor

    # 5. DataLoader'ın yaptığı gibi en başa batch boyutu (B=1) ekleyelim
    mixed_concat = mixed_concat.unsqueeze(0)
    person_mask = person_mask.unsqueeze(0)

    mixed_item = (mixed_concat, person_mask)

    print(f"Çıkarım başlatılıyor... (Kişi İndeksi: {person_idx}, Kıyafet: {os.path.basename(cloth_path)})")
    output = run_inference(mixed_item, num_steps=num_steps, cfg=cfg)

    return output


# Şimdilik çalışıp çalışmadığını test etmek için veri setinden rastgele bir kıyafet yolu alıyoruz:
sample_cloth_path = "/content/Adsız.jpg"

# Çıkarımı yap
output = try_on_custom_cloth(person_idx=103, cloth_path=sample_cloth_path, num_steps=50, cfg=2)

# Kaydet
save_image(output, fp="/content/custom_output.png")
print("✅ Sonuç /content/custom_output.png olarak kaydedildi.")

In [ ]:
# @title
import lpips
import torch
from tqdm import tqdm

# LPIPS modelini yükle (VGG ağı tabanlı)
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)
loss_fn_vgg.eval()

num_eval_samples = 64
total_lpips = 0.0
count = 0

print(f"Test seti üzerinden {num_eval_samples} örnek için LPIPS hesaplanıyor...")

for i, batch in enumerate(tqdm(test_loader, total=num_eval_samples)):
    if i >= num_eval_samples:
        break

    # Ground truth (orijinal) resmi al (concat tensor'ün sağ tarafı)
    concat_tensor = batch[0].to(device)
    _, _, H, W2 = concat_tensor.shape
    W = W2 // 2
    gt_image_01 = concat_tensor[:, :, :, W:] # [0, 1] aralığında orijinal resim

    # Çıkarımı çalıştır
    # Artık run_inference doğrudan sadece sağ tarafı döndürüyor.
    output_01 = run_inference(batch, num_steps=25, cfg=3.5,verbose=False)

    # LPIPS [-1, 1] aralığında değerler bekler, o yüzden dönüştürüyoruz
    gt_lpips = gt_image_01 * 2.0 - 1.0
    output_lpips = output_01 * 2.0 - 1.0

    # LPIPS değerini hesapla
    with torch.no_grad():
        lpips_val = loss_fn_vgg(output_lpips, gt_lpips)
        total_lpips += lpips_val.item()
        count += 1

avg_lpips = total_lpips / count
print(f"\n{count} örnek üzerinden Ortalama LPIPS Skoru: {avg_lpips:.4f}")
print("(LPIPS ne kadar düşükse, üretilen resim orijinaline o kadar çok benziyor demektir)")

In [ ]:
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=4)

In [ ]:
import lpips
import torch
import os
from tqdm import tqdm

# LPIPS modelini yükle (zaten yüklüyse tekrar yüklemez)

loss_fn_vgg = lpips.LPIPS(net='alex').to(device)
loss_fn_vgg.eval()

num_eval_samples = 16
cfg_values = [3.5]
step_values = [50]

# Test edilecek modeller
checkpoints = ["dream_cached_interarea_epoch_111_0.024398.pt"]

results = []
total_combinations = len(checkpoints) * len(cfg_values) * len(step_values)
print(f"Grid Search Başlıyor: {num_eval_samples} örnek üzerinde toplam {total_combinations} kombinasyon test edilecek.")

for ckpt_name in checkpoints:
    print(f"\n=======================================================")
    print(f"Model Yükleniyor: {ckpt_name}")
    print(f"=======================================================")

    # İlgili checkpoint'i UNet'e yükle (strict=False önemli, çünkü cross_attn yok)
    ckpt_path = os.path.join(save_dir, ckpt_name)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    unet.load_state_dict(ckpt["unet"], strict=False)
    unet.to("cuda")#az sonra bunu float32 yapcam

    for cfg in cfg_values:
        for steps in step_values:
            print(f"\n--- Test: Model={ckpt_name[:12]}..., CFG={cfg}, Steps={steps} ---")
            total_lpips = 0.0
            count = 0

            for i, batch in enumerate(tqdm(test_loader, total=num_eval_samples, leave=False)):
                if i >= num_eval_samples:
                    break

                concat_tensor = batch[0].to(device)
                _, _, H, W2 = concat_tensor.shape
                W = W2 // 2
                gt_image_01 = concat_tensor[:, :, :, W:]

                # Inference'ı çalıştır (Artık sadece sağ taraf dönüyor)
                output_01 = run_inference(batch, num_steps=steps, cfg=cfg, verbose=False)

                # LPIPS için [-1, 1] aralığına getir
                gt_lpips = gt_image_01 * 2.0 - 1.0
                output_lpips = output_01 * 2.0 - 1.0

                with torch.no_grad():
                    lpips_val = loss_fn_vgg(output_lpips, gt_lpips)
                    total_lpips += lpips_val.sum().item()
                    count += lpips_val.size(0)

            avg_lpips = total_lpips / count

            results.append({
                "model": ckpt_name,
                "cfg": cfg,
                "steps": steps,
                "lpips": avg_lpips
            })
            print(f"Sonuç -> LPIPS: {avg_lpips:.4f}")

# Sonuçları LPIPS değerine göre sırala (Düşük olması iyidir)
print("\n=== Grid Search Sonuçları Özeti (En İyiden En Kötüye) ===")
results_sorted = sorted(results, key=lambda x: x["lpips"])
for i, res in enumerate(results_sorted):
    print(f"{i+1:02d}. LPIPS: {res['lpips']:.4f} | Model: {res['model']} | CFG: {res['cfg']} | Steps: {res['steps']}")

In [ ]:
import os
import torch
from tqdm import tqdm

# Sadece pt dosyalarını ve belirli formatta olanları al
ckpt_files = [f for f in os.listdir(save_dir) if f.endswith('.pt') and "dream_cached_epoch_" in f]

# Dosya adından epoch numarasını çıkartan yardımcı fonksiyon
def get_epoch(filename):
    try:
        # Örn: dream_cached_epoch_110_0.025471.pt -> 110
        return int(filename.split('_')[3])
    except:
        return -1

# Epoch numarasına göre sırala
ckpt_files.sort(key=get_epoch)

# 10 aralıkla 10 model alacak şekilde ayarla
interval = 10
num_ckpts_to_average = 10
selected_ckpts = []

if len(ckpt_files) > 0:
    # Geriye doğru tarayarak interval'a uygun dosyaları seç
    for c in reversed(ckpt_files):
        ep = get_epoch(c)
        if ep == -1:
            continue
        if len(selected_ckpts) == 0 or (get_epoch(selected_ckpts[-1]) - ep) >= interval:
            selected_ckpts.append(c)
        if len(selected_ckpts) == num_ckpts_to_average:
            break

selected_ckpts.reverse() # Kronolojik sıralama için ters çevir

print(f"Seçilen {len(selected_ckpts)} checkpoint (10 epoch aralıklı):")
for c in selected_ckpts:
    print(c)

if len(selected_ckpts) > 0:
    avg_state_dict = None
    valid_ckpts_count = 0

    for i, ckpt_name in enumerate(tqdm(selected_ckpts, desc="EMA Hesaplanıyor")):
        ckpt_path = os.path.join(save_dir, ckpt_name)
        try:
            # Modeli doğrudan GPU'ya yükle
            ckpt = torch.load(ckpt_path, map_location="cuda")
            state_dict = ckpt["unet"]

            if avg_state_dict is None:
                # İlk checkpoint'i baz al
                avg_state_dict = {k: v.clone().float() for k, v in state_dict.items()}
            else:
                # Diğer checkpoint'leri üzerine topla
                for k in avg_state_dict.keys():
                    if k in state_dict:
                        avg_state_dict[k] += state_dict[k].float()

            # VRAM'in dolmaması için referansları sil ve belleği temizle
            del ckpt
            del state_dict
            torch.cuda.empty_cache()

            valid_ckpts_count += 1
        except Exception as e:
            print(f"\nHATA: {ckpt_name} dosyası bozuk veya okunamıyor. Atlanıyor... ({e})")

    if valid_ckpts_count > 0:
        # Toplamı geçerli checkpoint sayısına bölerek ortalamayı (SWA) bul
        for k in avg_state_dict.keys():
            avg_state_dict[k] = avg_state_dict[k] / valid_ckpts_count

        # unet tanımlı değilse yüklemeden önce oluştur
        if 'unet' not in globals():
            print("\nunet değişkeni tanımlı değil, model oluşturuluyor...")
            from diffusers import UNet2DConditionModel
            unet = UNet2DConditionModel.from_pretrained(
                "stable-diffusion-v1-5/stable-diffusion-inpainting",
                subfolder="unet"
            )
            if 'strip_cross_attention_permanently' in globals():
                unet = strip_cross_attention_permanently(unet)

        # Ortalama ağırlıkları modele yükle
        unet.load_state_dict(avg_state_dict, strict=False)
        unet.to("cuda")
        print(f"\n EMA (Averaged) model ağırlıkları {valid_ckpts_count} geçerli dosya ile başarıyla GPU üzerinde hesaplandı ve UNet'e yüklendi!")

        # İleride kullanmak için bu averaged checkpoint'i kaydedelim (kaydederken CPU'ya alalım)
        ema_save_path = os.path.join(save_dir, "ema_interval_10_epochs.pt")
        cpu_state_dict = {k: v.cpu() for k, v in avg_state_dict.items()}
        torch.save({"unet": cpu_state_dict}, ema_save_path)
        print(f" Averaged model şuraya kaydedildi: {ema_save_path}")
    else:
        print("\nOrtalaması alınacak sağlam checkpoint bulunamadı.")
else:
    print("\nOrtalaması alınacak uygun checkpoint bulunamadı.")

In [ ]:
import warnings
# Sinir bozucu uyarıları (warnings) gizle
warnings.filterwarnings("ignore")

import os
import torch
import lpips
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader

# VAE Slicing aktif et (VRAM tasarrufu)
vae.disable_slicing()

# Daha optimal batch size ile hızlı dataloader
eval_batch_size = 8
fast_eval_loader = DataLoader(test_ds, batch_size=eval_batch_size, shuffle=False, num_workers=4)

# Hızlı değerlendirme için 32 örnek
num_eval_samples = 64

# LPIPS modelini yükle (AlexNet daha hızlıdır)
loss_fn_alex = lpips.LPIPS(net='alex').to(device)
loss_fn_alex.eval()

# Klasör yolu
ckpts_dir = os.path.join(save_dir, "last_ckpts")
if not os.path.exists(ckpts_dir):
    print(f"Klasör bulunamadı: {ckpts_dir}. Lütfen yolu kontrol edin.")
else:
    # Dosyaları listele ve parse et
    all_ckpts = [f for f in os.listdir(ckpts_dir) if f.endswith('.pt') and "dream_cached_interarea_epoch_" in f]

    ckpt_data = []
    for f in all_ckpts:
        try:
            parts = f.split('_')
            epoch = int(parts[4])
            loss = float(parts[5].replace('.pt', ''))
            ckpt_data.append({'filename': f, 'epoch': epoch, 'loss': loss})
        except Exception as e:
            continue

    # Epoch'a göre sırala
    ckpt_data.sort(key=lambda x: x['epoch'])

    # 5'er atlayarak seç (örn: 0, 5, 10, ...)
    # İsterseniz bunu epoch % 5 == 0 olarak da değiştirebilirsiniz
    selected_ckpts = ckpt_data[::5]

    print(f"Toplam {len(ckpt_data)} model bulundu. 5 aralıkla {len(selected_ckpts)} model test edilecek.")

    results_epoch = []
    results_loss = []
    results_lpips = []

    for data in selected_ckpts:
        ckpt_path = os.path.join(ckpts_dir, data['filename'])

        # Modeli yükle
        ckpt = torch.load(ckpt_path, map_location="cpu")
        unet.load_state_dict(ckpt["unet"], strict=False)
        unet.to(device)

        total_lpips = 0.0
        count = 0

        with torch.no_grad():
            for i, batch in enumerate(tqdm(fast_eval_loader, desc=f"Eval Epoch {data['epoch']}", leave=False)):
                if i * eval_batch_size >= num_eval_samples:
                    break

                concat_tensor = batch[0].to(device)
                W2 = concat_tensor.shape[3]
                W = W2 // 2
                gt_image_01 = concat_tensor[:, :, :, W:]

                # Inference'ı çalıştır (hızlı olması için step=20 kullanıyoruz)
                output_01 = run_inference(batch, num_steps=20, cfg=3.5, verbose=False)

                gt_lpips = gt_image_01 * 2.0 - 1.0
                output_lpips = output_01 * 2.0 - 1.0

                lpips_val = loss_fn_alex(output_lpips, gt_lpips)
                total_lpips += lpips_val.sum().item()
                count += lpips_val.size(0)

        avg_lpips = total_lpips / count
        results_epoch.append(data['epoch'])
        results_loss.append(data['loss'])
        results_lpips.append(avg_lpips)
        print(f"Epoch {data['epoch']:03d} | Loss: {data['loss']:.6f} | LPIPS: {avg_lpips:.4f}")

        # VRAM temizliği
        torch.cuda.empty_cache()

    # ---- GRAFİK ÇİZİMİ ----
    if len(results_epoch) > 0:
        fig, ax1 = plt.subplots(figsize=(12, 6))

        color = 'tab:red'
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Training Loss', color=color)
        ax1.plot(results_epoch, results_loss, marker='o', color=color, label='Loss', linewidth=2)
        ax1.tick_params(axis='y', labelcolor=color)

        ax2 = ax1.twinx()
        color = 'tab:blue'
        ax2.set_ylabel('LPIPS (Daha düşük daha iyi)', color=color)
        ax2.plot(results_epoch, results_lpips, marker='s', color=color, label='LPIPS', linewidth=2)
        ax2.tick_params(axis='y', labelcolor=color)

        plt.title('Eğitim Kaybı (Loss) ve LPIPS Karşılaştırması')
        fig.tight_layout()
        ax1.grid(True, linestyle='--', alpha=0.6)
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
from google.colab import files
import os

# User provided LPIPS results (hardcoded to avoid re-running eval)
results_epoch = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 105, 110]
results_lpips = [0.2073, 0.1377, 0.1238, 0.1198, 0.1195, 0.1143, 0.1137, 0.1122, 0.1073, 0.1070, 0.1045, 0.1015, 0.1000, 0.0998, 0.1004, 0.1004, 0.0974, 0.1001, 0.0994, 0.0978, 0.0984, 0.0982, 0.0957]

# Extract losses directly from checkpoint filenames
ckpts_dir = os.path.join(save_dir, "last_ckpts")
ckpt_data = []
if os.path.exists(ckpts_dir):
    all_ckpts = [f for f in os.listdir(ckpts_dir) if f.endswith('.pt') and "dream_cached_interarea_epoch_" in f]
    for f in all_ckpts:
        try:
            parts = f.split('_')
            epoch = int(parts[4])
            loss = float(parts[5].replace('.pt', ''))
            ckpt_data.append({'epoch': epoch, 'loss': loss})
        except Exception as e:
            continue

# Sort by epoch
ckpt_data.sort(key=lambda x: x['epoch'])

if len(ckpt_data) > 0:
    all_epochs = [d['epoch'] for d in ckpt_data]
    all_losses = [d['loss'] for d in ckpt_data]

    fig, ax1 = plt.subplots(figsize=(12, 6))

    color = 'tab:red'
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss', color=color)
    # Plot all losses with smaller markers and thinner line
    ax1.plot(all_epochs, all_losses, marker='.', color=color, label='Loss', linewidth=1, markersize=5)
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('LPIPS (Lower is better)', color=color)
    # Plot LPIPS points
    ax2.plot(results_epoch, results_lpips, marker='s', color=color, label='LPIPS', linewidth=2)
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title('Comparison of Training Loss and LPIPS')
    fig.tight_layout()
    ax1.grid(True, linestyle='--', alpha=0.6)

    # Save the figure
    save_path = 'loss_lpips_comparison.png'
    plt.savefig(save_path, dpi=300)
    print(f"Plot saved to {save_path}")

    plt.show()

    # Download the file
    files.download(save_path)
else:
    print("No checkpoints found to plot the loss.")

In [ ]:
import random
import matplotlib.pyplot as plt
from diffusers import DDIMScheduler
import torch
from google.colab import files

# Ensure DDIM scheduler is used
inference_scheduler = DDIMScheduler.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="scheduler"
)

num_samples = 10
# User specified cloth indices (removed the duplicate 1528 to keep 10 samples)
cloth_indices = [557, 776, 65, 1394, 1400, 924, 1493, 154, 1528, 1496]

# Generate random person indices for all
person_indices = random.sample(range(len(test_ds)), num_samples)
# Set specific person 1910 for cloth 1400 (which is at index 4)
person_indices[4] = 1910

# We want 2 samples per row, so 5 rows and 6 columns
rows = 5
cols = 6
fig, axes = plt.subplots(rows, cols, figsize=(24, 5 * rows))

for i in range(num_samples):
    p_idx = person_indices[i]
    c_idx = cloth_indices[i]

    # Get data from dataset
    person_concat, person_mask = test_ds[p_idx]
    cloth_concat, _ = test_ds[c_idx]

    # Create mixed input
    mixed_concat = person_concat.clone()
    mixed_concat[:, :, :384] = cloth_concat[:, :, :384] # Swap the cloth (left side)

    # Add batch dimension
    mixed_item = (mixed_concat.unsqueeze(0), person_mask.unsqueeze(0))

    # Run inference
    print(f"Processing {i+1}/{num_samples} (Person: {p_idx}, Cloth: {c_idx})...")
    output = run_inference(mixed_item, num_steps=50, cfg=3.0, verbose=False)

    # Prepare images for plotting
    person_img = person_concat[:, :, 384:].permute(1, 2, 0).cpu().numpy() # Original Person (right side)
    cloth_img = cloth_concat[:, :, :384].permute(1, 2, 0).cpu().numpy()   # Target Cloth (left side)
    out_img = output.squeeze(0).permute(1, 2, 0).cpu().numpy()          # Generated Image

    # Determine grid position
    row_idx = i // 2
    col_offset = (i % 2) * 3

    # Plot
    axes[row_idx, col_offset + 0].imshow(person_img)
    axes[row_idx, col_offset + 0].set_title(f"Person ({p_idx})")
    axes[row_idx, col_offset + 0].axis("off")

    axes[row_idx, col_offset + 1].imshow(cloth_img)
    axes[row_idx, col_offset + 1].set_title(f"Target Cloth ({c_idx})")
    axes[row_idx, col_offset + 1].axis("off")

    axes[row_idx, col_offset + 2].imshow(out_img)
    axes[row_idx, col_offset + 2].set_title("Generated Output")
    axes[row_idx, col_offset + 2].axis("off")

plt.tight_layout()
plt.savefig("random_10_outputs.png", dpi=150)
print("Saved the figure as 'random_10_outputs.png'.")
plt.show()

# Download the file
files.download("random_10_outputs.png")